# Clean BERT fine-tuning — CMPE 586

Protocol: `docs/training-protocol.md`; configuration: `configs/training.json`.

This notebook prepares train/validation, measures a short GPU calibration, and can launch the clean baseline. Training logic lives in Python modules. Select a **GPU runtime** before starting. This notebook has not yet been executed on Colab.

The training files must first be present in the GitHub checkout. Full training has a separate switch below. Checkpoints and the shared initialization are stored in Drive; prepared inputs are cached there so a resumed run uses the same bytes. Calibration weights are discarded.


In [ ]:
import subprocess
import sys
from pathlib import Path

if sys.version_info[:2] not in ((3, 11), (3, 12)):
    raise RuntimeError("This project currently supports Python 3.11/3.12.")

PROJECT = Path("/content/cmpe586-turkish-punctuation")
REPOSITORY = "https://github.com/kckagancan/cmpe586-turkish-punctuation.git"
if not PROJECT.exists():
    subprocess.run(["git", "clone", REPOSITORY, str(PROJECT)], check=True)
if not (PROJECT / "scripts/train.py").exists():
    raise RuntimeError("The GitHub checkout does not contain the training files yet. Sync the reviewed project code first.")
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(PROJECT)], check=True)
sys.path.insert(0, str(PROJECT / "src"))
print("Project:", PROJECT)


## Runtime and persistent storage

Use Colab's GPU PyTorch installation. If installation changed already imported dependencies, restart the runtime before continuing. Resume requires matching package versions and GPU/runtime identity; the runner records them in `run.json`.


In [ ]:
import torch
from google.colab import drive

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime; the training runner requires CUDA.")
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/cmpe586-turkish-punctuation")
RUN_ROOT = DRIVE_ROOT / "training-v1"
INITIALIZATIONS = DRIVE_ROOT / "initializations/training-v1"
CACHE_ROOT = DRIVE_ROOT / "inputs/clean-v1"
RUN_ROOT.mkdir(parents=True, exist_ok=True)


## Prepare train and validation

Only the pinned train/validation CSVs are downloaded. The test split is not downloaded or prepared by these commands. Tokenizer assets are downloaded without punctuation-model weights. Existing prepared inputs are verified against the persistent cache.


In [ ]:
import json
import shutil
from tempfile import TemporaryDirectory
from transformers import AutoTokenizer
from turkish_punctuation.training_io import file_hashes, write_json

subprocess.run([sys.executable, "scripts/download_data.py", "--splits", "train", "validation"], cwd=PROJECT, check=True)
source = json.loads((PROJECT / "configs/tokenizer.json").read_text())
AutoTokenizer.from_pretrained(source["model_id"], revision=source["revision"], cache_dir=str(PROJECT / source["cache_directory"]), use_fast=True, trust_remote_code=False, token=False)
prepared = PROJECT / "data/processed/clean-v1"
if CACHE_ROOT.exists():
    hashes = json.loads((CACHE_ROOT / "files.json").read_text())
    if file_hashes(CACHE_ROOT / "prepared") != hashes:
        raise RuntimeError("Persistent prepared-input cache is incomplete or changed.")
    if not prepared.exists():
        shutil.copytree(CACHE_ROOT / "prepared", prepared)
    elif file_hashes(prepared) != hashes:
        raise RuntimeError("Local and persistent prepared inputs differ; use the matching input cache.")
else:
    if not prepared.exists():
        subprocess.run([sys.executable, "scripts/prepare_data.py", "--splits", "train", "validation"], cwd=PROJECT, check=True)
    CACHE_ROOT.parent.mkdir(parents=True, exist_ok=True)
    with TemporaryDirectory(prefix=".inputs-", dir=CACHE_ROOT.parent) as temporary:
        staging = Path(temporary) / "cache"
        shutil.copytree(prepared, staging / "prepared")
        write_json(staging / "files.json", file_hashes(staging / "prepared"))
        staging.rename(CACHE_ROOT)
print("Prepared inputs:", prepared)


## GPU calibration

The default is 30 optimizer updates from the shared initialization. The first update includes the longest clean records for a peak-memory observation; the first five updates are excluded from steady-state timing. The estimate covers training updates and excludes validation/checkpoint overhead. It is not a model-quality result or a noisy-input memory guarantee.

For a memory fallback, use batch/accumulation **4/8** or **2/16** and a new run tag. Use the same resolved pair for formal training.


In [ ]:
SEED = 586
BATCH_SIZE = 8
ACCUMULATION = 4
RUN_TAG = "v1"
CALIBRATION_OUTPUT = RUN_ROOT / f"calibration-clean-seed-{SEED}-{RUN_TAG}"
TRAIN_OUTPUT = RUN_ROOT / f"train-clean-seed-{SEED}-{RUN_TAG}"
BASE_COMMAND = [sys.executable, "scripts/train.py", "--config", "configs/training.json", "--seed", str(SEED), "--initialization-directory", str(INITIALIZATIONS), "--batch-size", str(BATCH_SIZE), "--gradient-accumulation", str(ACCUMULATION), "--download-model"]


In [ ]:
subprocess.run([*BASE_COMMAND, "--mode", "calibrate", "--calibration-steps", "30", "--output-directory", str(CALIBRATION_OUTPUT)], cwd=PROJECT, check=True)


In [ ]:
calibration = json.loads((CALIBRATION_OUTPUT / "calibration.json").read_text())
print(json.dumps(calibration, indent=2))


## Full clean baseline

After reviewing calibration, set `RUN_FULL_TRAINING=True` to run the three-epoch clean baseline. This starts again from the shared initialization. For an interrupted formal run, keep the same paths/settings and set `RESUME=True`; rerun setup/preparation as needed, then execute this cell without rerunning calibration. Completed runs are not overwritten.


In [ ]:
RUN_FULL_TRAINING = False
RESUME = False
if RUN_FULL_TRAINING:
    if not RESUME and not (CALIBRATION_OUTPUT / "calibration.json").exists():
        raise RuntimeError("Run the GPU calibration with these batch settings first.")
    command = [*BASE_COMMAND, "--mode", "train", "--output-directory", str(TRAIN_OUTPUT)]
    if RESUME:
        command.append("--resume")
    subprocess.run(command, cwd=PROJECT, check=True)
else:
    print("Full training has not been started.")


## Saved outputs

`run.json` records configuration, environment, input hashes and shared-initialization identity. `events.jsonl` logs loss/LR/step timing. `checkpoints/` holds resumable state. Each epoch has HF-loadable weights and clean validation predictions/metrics. `summary.json` is written after all epochs and identifies the best clean-validation checkpoint plus the final checkpoint. No test scores are produced.


In [ ]:
summary_path = TRAIN_OUTPUT / "summary.json"
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print("Best epoch:", summary["best_epoch"])
    print("Clean validation macro-F1:", summary["best_macro_f1"])
    print("Best model:", TRAIN_OUTPUT / summary["best_model"])
else:
    print("No completed clean training result yet.")
